In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, classification_report
)
from sklearn.preprocessing import label_binarize

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

!pip install micromlgen
try:
    from micromlgen import port
    MICROMLGEN_AVAILABLE = True
except ImportError:
    MICROMLGEN_AVAILABLE = False
    print("[INFO] micromlgen tidak tersedia. Install: pip install micromlgen")

import pickle

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

# Definisi threshold & label kelas
THRESHOLD_NORMAL  = 100   # ppm — di bawah ini: Normal
THRESHOLD_WASPADA = 400   # ppm — di bawah ini: Waspada, di atas: Bocor
CLASS_NAMES = ['Normal', 'Waspada', 'Bocor']
CLASS_COLORS = ['#2196F3', '#FF9800', '#F44336']

print("Library berhasil diimport.")
print(f"   Threshold Normal  : < {THRESHOLD_NORMAL} ppm")
print(f"   Threshold Waspada : {THRESHOLD_NORMAL} – {THRESHOLD_WASPADA} ppm")
print(f"   Threshold Bocor   : > {THRESHOLD_WASPADA} ppm")

In [ ]:
DATA_PATH = 'halolpg.csv'
df = pd.read_csv(DATA_PATH)

print(f"Dataset asli: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Label asli  : {df['Label'].value_counts().to_dict()}")
df.head(5)

In [ ]:
# Relabel Data Berdasarkan MQ-6 (ppm)

def relabel(ppm):
  if ppm < THRESHOLD_NORMAL:
    return 0
  elif ppm <= THRESHOLD_WASPADA:
    return 1
  else:
    return 2

df['Relabel'] = df['MQ-6 (ppm)'].apply(relabel)

print("Distribusi Label 3 Kelas:")
counts = df['Relabel'].value_counts().sort_index()
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i} ({name:7s}): {counts[i]:5d} sampel")
print(f"  Total          : {len(df):5d} sampel")

df.to_csv('data_relabel.csv', index=False)
print("CSV berhasil disimpan: data_relabeled.csv")

---
## Analisis Statistik Justifikasi Threshold
Bagian ini menampilkan analisis statistik deskriptif sebagai dasar penetapan threshold relabeling tiga kelas (Normal, Waspada, Bocor).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

THRESHOLD_NORMAL  = 100
THRESHOLD_WASPADA = 400
CLASS_NAMES  = ['Normal', 'Waspada', 'Bocor']
CLASS_COLORS = ['#2196F3', '#FF9800', '#F44336']

print('Library berhasil diimport.')

In [ ]:
# Muat dataset asli (label biner)
DATA_PATH = 'halolpg.csv'
df = pd.read_csv(DATA_PATH)

print(f'Dataset asli : {df.shape[0]} baris, {df.shape[1]} kolom')
print(f'Label asli   : {df["Label"].value_counts().to_dict()}')
df.head()

## 1. Analisis Statistik Deskriptif per Kelas Label Asli

In [ ]:
normal_data = df[df['Label'] == 0]['MQ-6 (ppm)']
bocor_data  = df[df['Label'] == 1]['MQ-6 (ppm)']

print('=' * 58)
print('  ANALISIS STATISTIK JUSTIFIKASI THRESHOLD')
print('=' * 58)

print('\n Kelas Normal (Label 0):')
print(f'   Jumlah sampel : {len(normal_data)} sampel')
print(f'   Mean          : {normal_data.mean():.2f} ppm')
print(f'   Std           : {normal_data.std():.2f} ppm')
print(f'   Min           : {normal_data.min():.2f} ppm')
print(f'   Persentil 25  : {normal_data.quantile(0.25):.2f} ppm')
print(f'   Persentil 50  : {normal_data.quantile(0.50):.2f} ppm')
print(f'   Persentil 75  : {normal_data.quantile(0.75):.2f} ppm')
print(f'   Persentil 90  : {normal_data.quantile(0.90):.2f} ppm')
print(f'   Persentil 95  : {normal_data.quantile(0.95):.2f} ppm')
print(f'   Max           : {normal_data.max():.2f} ppm  <-- acuan batas atas Normal')
print(f'   --> Threshold Normal ditetapkan : < {THRESHOLD_NORMAL} ppm')
print(f'       (pembulatan konservatif dari max = {normal_data.max():.2f} ppm,')
print(f'        memberikan margin keamanan terhadap noise sensor)')

print('\n Kelas Bocor (Label 1):')
print(f'   Jumlah sampel : {len(bocor_data)} sampel')
print(f'   Mean          : {bocor_data.mean():.2f} ppm')
print(f'   Std           : {bocor_data.std():.2f} ppm')
print(f'   Min           : {bocor_data.min():.2f} ppm')
print(f'   Persentil 10  : {bocor_data.quantile(0.10):.2f} ppm')
print(f'   Persentil 25  : {bocor_data.quantile(0.25):.2f} ppm  <-- acuan batas bawah Bocor')
print(f'   Persentil 50  : {bocor_data.quantile(0.50):.2f} ppm')
print(f'   Persentil 75  : {bocor_data.quantile(0.75):.2f} ppm')
print(f'   Max           : {bocor_data.max():.2f} ppm')
print(f'   --> Threshold Bocor ditetapkan  : > {THRESHOLD_WASPADA} ppm')
print(f'       (pembulatan dari persentil-25 = {bocor_data.quantile(0.25):.2f} ppm,')
print(f'        memastikan >= 75% sampel bocor terklasifikasi dengan benar)')

print('\n Zona Waspada (buffer transisi):')
print(f'   Rentang       : {THRESHOLD_NORMAL} - {THRESHOLD_WASPADA} ppm')
print(f'   Max Normal    : {normal_data.max():.2f} ppm')
print(f'   Min Bocor     : {bocor_data.min():.2f} ppm')
gap = bocor_data.min() - normal_data.max()
print(f'   Gap antar kelas asli : {gap:.2f} ppm')
print(f'   --> Zona Waspada merepresentasikan kondisi transisi')
print(f'       antara operasi normal dan kebocoran terdeteksi.')
print('=' * 58)

## 2. Visualisasi Distribusi Konsentrasi Gas per Kelas Label Asli

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Histogram distribusi per kelas ---
ax1 = axes[0]
ax1.hist(normal_data, bins=50, color='#2196F3', alpha=0.7, label='Normal (Label 0)', edgecolor='white', linewidth=0.4)
ax1.hist(bocor_data,  bins=50, color='#F44336', alpha=0.7, label='Bocor (Label 1)',  edgecolor='white', linewidth=0.4)
ax1.axvline(THRESHOLD_NORMAL,  color='#1565C0', linestyle='--', linewidth=1.5, label=f'Threshold Normal = {THRESHOLD_NORMAL} ppm')
ax1.axvline(THRESHOLD_WASPADA, color='#B71C1C', linestyle='--', linewidth=1.5, label=f'Threshold Bocor = {THRESHOLD_WASPADA} ppm')
ax1.axvline(normal_data.max(), color='#0D47A1', linestyle=':',  linewidth=1.2, label=f'Max Normal = {normal_data.max():.1f} ppm')
ax1.axvline(bocor_data.quantile(0.25), color='#880E4F', linestyle=':', linewidth=1.2, label=f'P25 Bocor = {bocor_data.quantile(0.25):.1f} ppm')
ax1.set_xlabel('Konsentrasi Gas MQ-6 (ppm)', fontsize=11)
ax1.set_ylabel('Frekuensi', fontsize=11)
ax1.set_title('Distribusi Konsentrasi Gas\nper Kelas Label Asli', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8)
ax1.set_xlim(left=0)

# --- Plot 2: Boxplot per kelas ---
ax2 = axes[1]
bp = ax2.boxplot(
    [normal_data, bocor_data],
    labels=['Normal\n(Label 0)', 'Bocor\n(Label 1)'],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2)
)
bp['boxes'][0].set_facecolor('#90CAF9')
bp['boxes'][1].set_facecolor('#EF9A9A')
ax2.axhline(THRESHOLD_NORMAL,  color='#1565C0', linestyle='--', linewidth=1.5, label=f'Threshold Normal = {THRESHOLD_NORMAL} ppm')
ax2.axhline(THRESHOLD_WASPADA, color='#B71C1C', linestyle='--', linewidth=1.5, label=f'Threshold Bocor = {THRESHOLD_WASPADA} ppm')
ax2.set_ylabel('Konsentrasi Gas MQ-6 (ppm)', fontsize=11)
ax2.set_title('Boxplot Konsentrasi Gas\nper Kelas Label Asli', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Analisis Statistik Justifikasi Threshold Relabeling', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('analisis_threshold.png', bbox_inches='tight', dpi=150)
plt.show()
print('Grafik disimpan: analisis_threshold.png')

## 3. Visualisasi Zona Klasifikasi Setelah Relabeling

In [ ]:
# Relabel data
def relabel(ppm):
    if ppm < THRESHOLD_NORMAL:
        return 0
    elif ppm <= THRESHOLD_WASPADA:
        return 1
    else:
        return 2

df['Relabel'] = df['MQ-6 (ppm)'].apply(relabel)

fig, ax = plt.subplots(figsize=(12, 5))

# Zona warna background
ax.axvspan(0,                  THRESHOLD_NORMAL,  alpha=0.12, color='#2196F3', label='Zona Normal')
ax.axvspan(THRESHOLD_NORMAL,   THRESHOLD_WASPADA, alpha=0.12, color='#FF9800', label='Zona Waspada')
ax.axvspan(THRESHOLD_WASPADA,  df['MQ-6 (ppm)'].max() * 1.05, alpha=0.12, color='#F44336', label='Zona Bocor')

# Histogram per kelas relabel
for i, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    subset = df[df['Relabel'] == i]['MQ-6 (ppm)']
    ax.hist(subset, bins=60, color=color, alpha=0.75, label=f'{name} ({len(subset)} sampel)', edgecolor='white', linewidth=0.3)

# Garis threshold
ax.axvline(THRESHOLD_NORMAL,  color='#1565C0', linestyle='--', linewidth=2, label=f'Threshold Normal = {THRESHOLD_NORMAL} ppm')
ax.axvline(THRESHOLD_WASPADA, color='#B71C1C', linestyle='--', linewidth=2, label=f'Threshold Bocor = {THRESHOLD_WASPADA} ppm')

# Anotasi
ax.annotate(f'Max Normal\n{normal_data.max():.1f} ppm',
            xy=(normal_data.max(), ax.get_ylim()[1] * 0.6),
            xytext=(normal_data.max() + 80, ax.get_ylim()[1] * 0.7),
            arrowprops=dict(arrowstyle='->', color='#0D47A1'),
            fontsize=9, color='#0D47A1')

ax.annotate(f'P25 Bocor\n{bocor_data.quantile(0.25):.1f} ppm',
            xy=(bocor_data.quantile(0.25), ax.get_ylim()[1] * 0.4),
            xytext=(bocor_data.quantile(0.25) + 100, ax.get_ylim()[1] * 0.55),
            arrowprops=dict(arrowstyle='->', color='#880E4F'),
            fontsize=9, color='#880E4F')

ax.set_xlabel('Konsentrasi Gas MQ-6 (ppm)', fontsize=11)
ax.set_ylabel('Frekuensi', fontsize=11)
ax.set_title('Distribusi Kelas Setelah Relabeling\ndengan Zona Klasifikasi dan Acuan Statistik Threshold', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(left=0)

plt.tight_layout()
plt.savefig('zona_relabeling.png', bbox_inches='tight', dpi=150)
plt.show()
print('Grafik disimpan: zona_relabeling.png')

## 4. Ringkasan Justifikasi Threshold

In [ ]:
print('=' * 58)
print('  RINGKASAN JUSTIFIKASI THRESHOLD')
print('=' * 58)
print(f'''
Threshold 100 ppm (batas Normal):
  - Nilai maksimum label Normal asli = {normal_data.max():.2f} ppm
  - Dibulatkan ke 100 ppm (margin keamanan noise sensor)
  - Seluruh {len(normal_data)} sampel Normal berada di bawah nilai ini

Threshold 400 ppm (batas Bocor):
  - Persentil ke-25 label Bocor asli = {bocor_data.quantile(0.25):.2f} ppm
  - Dibulatkan ke 400 ppm (nilai bersih & konsisten operasional)
  - Memastikan >= 75% dari {len(bocor_data)} sampel Bocor terklasifikasi tepat

Zona Waspada (100-400 ppm):
  - Buffer transisi antara Normal dan Bocor
  - Berfungsi sebagai peringatan dini sebelum kondisi bocor penuh
  - Mengurangi false alarm akibat lonjakan konsentrasi sementara
''')
print('Distribusi kelas setelah relabeling:')
counts = df['Relabel'].value_counts().sort_index()
for i, name in enumerate(CLASS_NAMES):
    pct = counts[i] / len(df) * 100
    print(f'  {name:7s} (Label {i}): {counts[i]:5d} sampel ({pct:.1f}%)')
print(f'  Total           : {len(df):5d} sampel')
print('=' * 58)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data_relabel.csv")

df["Datetime"] = pd.to_datetime(
    df["Tanggal"] + " " + df["Waktu"],
    dayfirst=True,
    errors="coerce"
)

data_grafik = df.iloc[5400:5500]

plt.figure(figsize=(12, 5))

plt.fill_between(
    data_grafik["Datetime"],
    data_grafik["MQ-6 (ppm)"],
    alpha=0.5,
    label="Konsentrasi Gas LPG"
)

plt.plot(
    data_grafik["Datetime"],
    data_grafik["MQ-6 (ppm)"],
    marker="o",
    linewidth=2
)

plt.title("Grafik Konsentrasi Gas LPG terhadap Waktu")
plt.xlabel("Waktu")
plt.ylabel("Konsentrasi Gas LPG (ppm)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

plt.savefig("grafik_konsentrasi_gas.png", dpi=300, bbox_inches='tight')

In [ ]:
# Distribusi Kelas
counts = df['Relabel'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(CLASS_NAMES, counts, color=CLASS_COLORS)
axes[0].set_title("Distribusi Kelas", fontweight = 'bold', fontsize = 13)
axes[0].set_ylabel("Jumlah Data")
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=CLASS_NAMES, colors=CLASS_COLORS,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporsi Kelas', fontweight='bold', fontsize=13)

plt.suptitle('Distribusi Dataset — Deteksi Kebocoran Gas LPG',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi ppm dengan Batas Threshold

plt.figure(figsize=(12, 4))

for i, (label, name, color) in enumerate(zip([0,1,2], CLASS_NAMES, CLASS_COLORS)):
    subset = df[df['Relabel'] == label]['MQ-6 (ppm)']
    plt.hist(subset, bins=50, alpha=0.6, color=color, label=f'{name} (n={len(subset)})')

plt.axvline(THRESHOLD_NORMAL,  color='orange', linestyle='--', lw=2,
            label=f'Batas Normal–Waspada ({THRESHOLD_NORMAL} ppm)')
plt.axvline(THRESHOLD_WASPADA, color='red',    linestyle='--', lw=2,
            label=f'Batas Waspada–Bocor ({THRESHOLD_WASPADA} ppm)')

plt.xlabel('MQ-6 (ppm)', fontsize=11)
plt.ylabel('Frekuensi')
plt.title('Distribusi MQ-6 (ppm) per Kelas dengan Threshold', fontweight='bold', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
FEATURE_COLS = ['Temperature', 'Humidity', 'MQ-6 (ppm)', 'ADC']
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for i, feat in enumerate(FEATURE_COLS):
    data_per_class = [df[df['Relabel'] == c][feat].values for c in [0,1,2]]
    bp = axes[i].boxplot(data_per_class, labels=CLASS_NAMES, patch_artist=True,
                         medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], CLASS_COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_ylabel('Nilai')

plt.suptitle('Boxplot Fitur per Kelas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Statistik Deskriptif

print("Statistik Deskriptif per Kelas:")
df.groupby('Relabel')[FEATURE_COLS].mean().round(3).rename(
    index={0:'Normal', 1:'Waspada', 2:'Bocor'})

In [ ]:
# Pra Pemrosesan Data

X = df[FEATURE_COLS].copy()
y = df['Relabel'].copy()

# Tambah noise realistis sensor
np.random.seed(RANDOM_STATE)
X['MQ-6 (ppm)'] += np.random.normal(0, 0.05 * X['MQ-6 (ppm)'].std(), size=len(X))
X['Temperature'] += np.random.normal(0, 0.30, size=len(X))
X['Humidity']    += np.random.normal(0, 1.00, size=len(X))
X['MQ-6 (ppm)']  = X['MQ-6 (ppm)'].clip(lower=0)
X['Humidity']    = X['Humidity'].clip(0, 100)

# Train-test split 80:20 stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Data latih : {X_train.shape[0]} sampel")
print(f"Data uji   : {X_test.shape[0]} sampel")
print(f"\nDistribusi kelas data latih:")
print(pd.Series(y_train).value_counts().sort_index().rename(
    {0:'Normal', 1:'Waspada', 2:'Bocor'}))

In [ ]:
# Imbalanced Dataset

# SMOTE untuk multiclass: oversample kelas minoritas
over  = SMOTE(sampling_strategy='not majority', random_state=RANDOM_STATE)
under = RandomUnderSampler(sampling_strategy='not minority', random_state=RANDOM_STATE)

pipeline_resample = ImbPipeline(steps=[('over', over), ('under', under)])
X_train_res, y_train_res = pipeline_resample.fit_resample(X_train, y_train)

before = pd.Series(y_train).value_counts().sort_index()
after  = pd.Series(y_train_res).value_counts().sort_index()

print(f"Sebelum: {dict(zip(CLASS_NAMES, before.values))}")
print(f"Sesudah: {dict(zip(CLASS_NAMES, after.values))}")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, title in zip(axes, [before, after],
                            ['Sebelum Resampling', 'Sesudah Resampling (SMOTE + Undersampling)']):
    bars = ax.bar(CLASS_NAMES, data.values, color=CLASS_COLORS, edgecolor='black', width=0.45)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Jumlah Data')
    for bar, v in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(v), ha='center', fontweight='bold')
plt.suptitle('Distribusi Kelas Data Latih', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pelatihan Random Forest

rf_model = RandomForestClassifier(
    n_estimators      = 20,
    max_depth         = 10,
    min_samples_split = 5,
    min_samples_leaf  = 2,
    max_features      = 'sqrt',
    class_weight      = 'balanced',
    random_state      = RANDOM_STATE,
    n_jobs            = -1
)

rf_model.fit(X_train_res, y_train_res)
print("Model Random Forest Berhasil Dilatih.")

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf_model, X_train_res, y_train_res,
                             cv=cv, scoring='f1_macro', n_jobs=-1)
print(f"\nCross-Validation F1-Macro (5-Fold):")
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"  Mean ± Std: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# Evaluasi Model dan Confusion Matrix

y_pred      = rf_model.predict(X_test)
y_pred_prob = rf_model.predict_proba(X_test)

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=True, cmap='Blues',)
ax.set_title('Confusion Matrix', fontweight='bold', fontsize=13)
ax.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Metrik Evaluasi

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall    = recall_score(y_test, y_pred, average='weighted')
f1        = f1_score(y_test, y_pred, average='weighted')
f1_macro  = f1_score(y_test, y_pred, average='macro')

# ROC-AUC multiclass (One-vs-Rest)
y_bin  = label_binarize(y_test, classes=[0,1,2])
roc_auc = roc_auc_score(y_bin, y_pred_prob, multi_class='ovr', average='macro')

# FAR & MDR per kelas
print(f"  METRIK EVALUASI")
print(" ")
print(f"  Akurasi              : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Presisi (weighted)   : {precision:.4f}")
print(f"  Recall  (weighted)   : {recall:.4f}")
print(f"  F1-Score (weighted)  : {f1:.4f}")
print(f"  F1-Score (macro)     : {f1_macro:.4f}")
print(f"  ROC-AUC (macro OvR)  : {roc_auc:.4f}")
print("-" * 55)

# FAR & MDR per kelas dari confusion matrix
print("  False Alarm Rate & Missed Detection per Kelas:")
for i, name in enumerate(CLASS_NAMES):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    FP = cm[:, i].sum() - TP
    TN = cm.sum() - TP - FP - FN
    FAR_i = FP / (FP + TN) if (FP + TN) > 0 else 0
    MDR_i = FN / (FN + TP) if (FN + TP) > 0 else 0
    print(f"    {name:7s} → FAR: {FAR_i:.4f} ({FAR_i*100:.2f}%) | MDR: {MDR_i:.4f} ({MDR_i*100:.2f}%)")

print("=" * 55)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ROC Curve

from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 5))
for i, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_prob[:, i])
    auc_score   = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc_score:.4f})')

plt.plot([0,1],[0,1], 'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.title('ROC Curve per Kelas', fontweight='bold', fontsize=13)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Baseline Comparison

# Skenario A (Threshold Statis)

y_baseline = X_test['MQ-6 (ppm)'].apply(relabel)

acc_base = accuracy_score(y_test, y_baseline)
f1_base  = f1_score(y_test, y_baseline, average='weighted', zero_division=0)
f1m_base = f1_score(y_test, y_baseline, average='macro', zero_division=0)

cm_base = confusion_matrix(y_test, y_baseline)
FAR_base_list, MDR_base_list = [], []
for i in range(3):
    TP = cm_base[i,i]; FN = cm_base[i,:].sum()-TP
    FP = cm_base[:,i].sum()-TP; TN = cm_base.sum()-TP-FP-FN
    FAR_base_list.append(FP/(FP+TN) if (FP+TN)>0 else 0)
    MDR_base_list.append(FN/(FN+TP) if (FN+TP)>0 else 0)

FAR_rf_list, MDR_rf_list = [], []
for i in range(3):
    TP = cm[i,i]; FN = cm[i,:].sum()-TP
    FP = cm[:,i].sum()-TP; TN = cm.sum()-TP-FP-FN
    FAR_rf_list.append(FP/(FP+TN) if (FP+TN)>0 else 0)
    MDR_rf_list.append(FN/(FN+TP) if (FN+TP)>0 else 0)

print(" ")
print(" PERBANDINGAN SKENARIO A (Threshold Statis) vs SKENARIO B (RANDOM FOREST)")
print("")
print(f"{'Metrik':<30} {'Skenario A':>15} {'Skenario B':>15}")
print("-" * 65)
print(f"{'Akurasi':<30} {acc_base:>15.4f} {accuracy:>15.4f}")
print(f"{'F1-Score (weighted)':<30} {f1_base:>15.4f} {f1:>15.4f}")
print(f"{'F1-Score (macro)':<30} {f1m_base:>15.4f} {f1_macro:>15.4f}")
print("-" * 65)
for i, name in enumerate(CLASS_NAMES):
    print(f"FAR {name:<26} {FAR_base_list[i]:>15.4f} {FAR_rf_list[i]:>15.4f}")
print("-" * 65)
for i, name in enumerate(CLASS_NAMES):
    print(f"MDR {name:<26} {MDR_base_list[i]:>15.4f} {MDR_rf_list[i]:>15.4f}")
print("=" * 65)

In [ ]:
# Visualisasi FAR

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# FAR
b1 = axes[0].bar(x - width/2, FAR_base_list, width, label='Skenario A (Baseline)', color='#90CAF9', edgecolor='black')
b2 = axes[0].bar(x + width/2, FAR_rf_list,   width, label='Skenario B (RF)',        color='#EF9A9A', edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASS_NAMES)
axes[0].set_ylabel('False Alarm Rate')
axes[0].set_title('False Alarm Rate per Kelas\n(lebih rendah = lebih baik)', fontweight='bold')
axes[0].legend()
for bar in list(b1)+list(b2):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{bar.get_height():.4f}', ha='center', fontsize=8)

# MDR
b3 = axes[1].bar(x - width/2, MDR_base_list, width, label='Skenario A (Baseline)', color='#90CAF9', edgecolor='black')
b4 = axes[1].bar(x + width/2, MDR_rf_list,   width, label='Skenario B (RF)',        color='#EF9A9A', edgecolor='black')
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES)
axes[1].set_ylabel('Missed Detection Rate')
axes[1].set_title('Missed Detection Rate per Kelas\n(lebih rendah = lebih baik)', fontweight='bold')
axes[1].legend()
for bar in list(b3)+list(b4):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{bar.get_height():.4f}', ha='center', fontsize=8)

plt.suptitle('Perbandingan Skenario A vs B — FAR & MDR per Kelas',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance

importances = rf_model.feature_importances_
std_imp     = np.std([t.feature_importances_ for t in rf_model.estimators_], axis=0)

feat_imp_df = pd.DataFrame({'Fitur': FEATURE_COLS, 'Importance': importances, 'Std': std_imp})
feat_imp_df = feat_imp_df.sort_values('Importance', ascending=False)

print("Feature Importance:")
print(feat_imp_df.to_string(index=False))

plt.figure(figsize=(8, 4))
bar_colors = ['#F44336' if f in ['MQ-6 (ppm)', 'ADC'] else '#2196F3' for f in feat_imp_df['Fitur']]
bars = plt.barh(feat_imp_df['Fitur'], feat_imp_df['Importance'],
                xerr=feat_imp_df['Std'], color=bar_colors, edgecolor='black', capsize=4)
plt.xlabel('Importance (Mean Decrease Impurity)', fontsize=11)
plt.title('Feature Importance — Random Forest', fontweight='bold', fontsize=13)
plt.gca().invert_yaxis()
for bar, val in zip(bars, feat_imp_df['Importance']):
    plt.text(val+0.005, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
if MICROMLGEN_AVAILABLE:
    c_code = port(rf_model, classmap={0: 'Normal', 1: 'Waspada', 2: 'Bocor'})
    with open('rf_model_3class_esp32.h', 'w') as f:
        f.write(c_code)
    print("✅ Model diekspor ke: rf_model_3class_esp32.h")
    print('\n'.join(c_code.split('\n')[:20]))

In [ ]:
# Ringkasan

print(" ")
print("  RINGKASAN")
print("  Navira Arditha Aulia — 2209106010")
print(" ")
counts = df['Relabel'].value_counts().sort_index()
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:7s} (Label {i}): {counts[i]} sampel  | threshold: ", end='')
    if i == 0: print(f"< {THRESHOLD_NORMAL} ppm")
    elif i == 1: print(f"{THRESHOLD_NORMAL}–{THRESHOLD_WASPADA} ppm")
    else: print(f"> {THRESHOLD_WASPADA} ppm")
print("-" * 60)
print(f"  Akurasi              : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  F1-Score (weighted)  : {f1:.4f}")
print(f"  F1-Score (macro)     : {f1_macro:.4f}")
print(f"  ROC-AUC (macro OvR)  : {roc_auc:.4f}")
print("-" * 60)
for i, name in enumerate(CLASS_NAMES):
    far_red = (FAR_base_list[i]-FAR_rf_list[i])/FAR_base_list[i]*100 if FAR_base_list[i]>0 else 0
    print(f"  FAR {name:7s}: Baseline={FAR_base_list[i]:.4f} → RF={FAR_rf_list[i]:.4f} (↓{far_red:.1f}%)")
print("=" * 60)